In [ ]:
import pandas as pd
import geopandas as gpd
import pathlib
import mapclassify as mc
import seaborn as sns
import matplotlib.pyplot as plt

## 경로설정

In [ ]:
BASE_PATH = pathlib.Path().resolve()

if BASE_PATH.name == "notebooks":
    BASE_PATH = BASE_PATH.parent
elif BASE_PATH.name != "analysis_table" and (BASE_PATH / "analysis_table").exists():
    BASE_PATH = BASE_PATH / "analysis_table"

INPUT_PATH = BASE_PATH / "data" / "input"
OUTPUT_PATH = BASE_PATH / "data" / "output"

DATA_PATH = INPUT_PATH / "raw"
PRO_PATH = OUTPUT_PATH

PRO_PATH.mkdir(parents=True, exist_ok=True)


## 데이터 불러오기
- 100m 격자 데이터
- 행정동 경계
- 구 경계
- .shp  -> geometry
- .dbf  -> GRID_CD 같은 속성 컬럼
- .prj  -> 좌표계
- .shx  -> 공간 인덱스


In [ ]:
# 100m 격자데이터
GRID_PATH = DATA_PATH / 'spatial' / 'grid' / 'source'
grid_100 = gpd.read_file(GRID_PATH / 'grid_dasa_100M.shp')
display(grid_100.head())


In [ ]:
print(f"그리드 100 CRS: {grid_100.crs}") # EPSG 5179
print(f"그리드 100 칼럼: {grid_100.columns}") # GRID_CD, geometry
print(f"그리드 100 지오메트리 타입: {grid_100.geometry.geom_type.unique()}") #polygon
print(f"그리드 100 레코드 수 : {len(grid_100)}") # 824647개
display(
    grid_100.head
)

fig, ax = plt.subplots(dpi=200)
grid_100[:5000].plot(
    ax = ax,
    edgecolor='black',
    linewidth=1.2,
    facecolor='None',
)
plt.show()

In [ ]:
# 행정동 경계 불러오기
BOUNDARY_PATH = DATA_PATH / 'spatial' / 'boundary'

hjd = gpd.read_file(BOUNDARY_PATH / 'BND_ADM_DONG_PG.shp', encoding = 'cp949')

print(f"행정동 crs: {hjd.crs}") # EPSG 5186
print(f"행정동 데이터수 {len(hjd)}") #3559개
print(f"행정동 칼럼 {hjd.columns}") # BASE_DATE ,ADM_CD(행정동 코드), ADM_NM (행정동 이름)
print(f"행정동 geomtype {hjd.geometry.geom_type.unique()}") # 폴리곤, 멀티폴리곤
display(hjd.head()) # 전국 행정동 코드, 서울 지역 클립 필요 

hjd.plot()


In [ ]:
IMAGE_PATH = BASE_PATH / "image"
IMAGE_PATH.mkdir(parents=True, exist_ok=True)

hjd_code = pd.read_excel(BOUNDARY_PATH / "BND_ADM_DONG_PG_geocode.xlsx", sheet_name=0, header=1)

hjd_code.head() # 시도코드 시군구코드 읍면동 코드 합칠필요, 시군구명칭, 읍면동 명칭 들어갈 필요

hjd_code = hjd_code[["시도코드", "시군구코드", "시군구명칭", "읍면동코드", "읍면동명칭"]]

# 코드 만들기
hjd_code[["시도코드", "시군구코드", "읍면동코드"]] = hjd_code[["시도코드", "시군구코드", "읍면동코드"]].astype(str)
hjd_code["ADM_CD"] = (
    hjd_code["시도코드"].astype(str).str.zfill(2)
    + hjd_code["시군구코드"].astype(str).str.zfill(3)
    + hjd_code["읍면동코드"].astype(str).str.zfill(3)
)

display(
    hjd_code.head()
)

# 행정동 코드 개수확인
print(hjd["ADM_CD"].str.len().value_counts())
print(hjd_code["ADM_CD"].str.len().value_counts())

# 테이블 결합
hjd_join = hjd.copy()

# 칼럼기준 merge / index 기준 join
hjd_join = hjd_join.merge(
    hjd_code,
    on = "ADM_CD",
    how = 'left'
)

hjd_seoul = hjd_join[hjd_join["시도코드"] == '11']
hjd_seoul = hjd_seoul[["ADM_CD", "시군구명칭", "ADM_NM", "geometry"]]
hjd_seoul = hjd_seoul.rename(
    columns = 
    {"ADM_CD": "행정동코드", 
     "시군구명칭": "시군구",
     "ADM_NM": "행정동"}
)

# 결과 확인
print(hjd_seoul.shape)
print(hjd_seoul.isna().sum())
print(len(hjd_seoul))

# 시각화
plt.rcParams['font.family'] = 'Noto Sans KR'

fig, ax = plt.subplots(figsize=(6, 6))
hjd_seoul.plot(
    ax=ax,
    edgecolor = 'black',
    facecolor = 'None',
    linewidth=0.8,
    alpha=0.7
)
ax.set_title("서울시 행정동 경계")
ax.set_axis_off()
fig.savefig(
    IMAGE_PATH / "행정동경계.png",
    dpi = 240,
    bbox_inches = 'tight',
    pad_inches = 0.1
)
plt.show()
# 시군구별 경계 합치기
sgg = hjd_seoul.dissolve(by="시군구").reset_index()

fig, ax = plt.subplots(figsize=(6, 6))
sgg.plot(
    ax=ax,
    edgecolor = 'black',
    facecolor = 'None',
    linewidth=0.8,
    alpha=0.7
    )
ax.set_title("서울시 시군구 경계")
ax.set_axis_off()

# 시군구 이름 표시
for idx, row in sgg.iterrows():
    point = row.geometry.representative_point()
    ax.text(
        point.x,
        point.y,
        row["시군구"],
        fontsize=9,
        ha="center",
        va="center"
    )
fig.savefig(
    IMAGE_PATH / "서울시_시군구경계.png",
    bbox_inches = 'tight',
    pad_inches = 0.1
)
plt.show()

# 성공!

In [ ]:
# 그리드 100 - 행정동 붙이기
grid_100.crs
hjd_seoul.crs

# EPSG:5179로 맞추기
hjd_seoul = hjd_seoul.to_crs(grid_100.crs)
assert hjd_seoul.crs == grid_100.crs, "CRS가 맞지 않음"
print(hjd_seoul.crs)

# 그리드 데이터를 시군구, 행정동 기준으로 공간 결합, 테이블 속성 붙이기

# 그리드 중심점 구하기
grid_100["중심점"] = grid_100.centroid
grid_100 = grid_100.set_geometry("중심점")

# 그리드 중심점을 within으로 sjoin 공간결합
grid_join = grid_100.sjoin(
    hjd_seoul,
    how = 'inner',
    predicate = 'within'
)

# 행정동 결합
display(grid_100.head())
display(grid_join.head())
display(hjd_seoul.head())

grid_hjd = grid_100.merge(
    grid_join[["GRID_CD", "행정동코드", "시군구", "행정동"]],
    on = "GRID_CD",
    how = 'inner',
)

# geometry 칼럼 옮기기
grid_hjd = grid_hjd.set_geometry("geometry")

# 결과 확인
print(f"그리드_행정동 crs: {grid_hjd.crs}")
print(f"그리드_행정동 데이터수 {len(grid_hjd)}")
print(f"그리드_행정동 칼럼 {grid_hjd.columns}") 
print(f"그리드_행정동 geomtype {grid_hjd.geometry.geom_type.unique()}") 
# 그리드_행정동 crs: EPSG:5179
# 그리드_행정동 데이터수 60528
# 그리드_행정동 칼럼 Index(['GRID_CD', 'geometry', '중심점', '행정동코드', '시군구', '행정동'], dtype='str')
# 그리드_행정동 geomtype ['Polygon']

In [ ]:
# 데이터 품질 확인 
print(grid_hjd.isna().sum())
print(grid_hjd["GRID_CD"].duplicated().sum())
print(grid_hjd["시군구"].nunique())
print(grid_hjd["행정동코드"].nunique())
# 구 : 25개, 행정동: 426개로 문제 없음 


# 시각화
fig, ax = plt.subplots(figsize=(8, 8))

grid_hjd.plot(
    ax=ax,
    facecolor="lightgray",
    edgecolor="none"
)

hjd_seoul.boundary.plot(
    ax=ax,
    color="black",
    linewidth=0.5
)
plt.show()

In [ ]:
# 기본 데이터 테이블 저장
grid_hjd_save = grid_hjd.copy()
grid_hjd_save["중심점_x"] = grid_hjd_save["중심점"].x
grid_hjd_save["중심점_y"] = grid_hjd_save["중심점"].y
grid_hjd_save = grid_hjd_save.drop(columns = "중심점")

grid_hjd_save.to_file(PRO_PATH / "서울시_격자_100m_행정동_기본테이블.gpkg",
                      driver='GPKG')

## 100m 격자 테이블에 500m 격자와 인구 통계 데이터 결합

In [ ]:
grid = gpd.read_file(PRO_PATH / "서울시_격자_100m_행정동_기본테이블.gpkg")
grid.head()

grid.plot(
    facecolor='None'
)

In [ ]:
a = gpd.read_file(OUTPUT_PATH / "인천경기_외부25km_100m_추정인구.gpkg", 
                  source = 'gpkg')
a.head()

a["추정_인구수"].describe()

In [ ]:
a.plot(
)